# Modeling
This notebook implements the recommendation system algorithms. In this first phase, we develop a Content-Based Filtering baseline using movie genre features and Cosine Similarity.

## 1. Content-Based Filtering (Genres)

### 1.1 Data Loading & Feature Selection

In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df_movies = pd.read_csv('../datasets/processed/movies_cleaned.csv')
df_movies.head()

,movieId,search_title,title,year,action,adventure,animation,children,comedy,crime,...,film-noir,horror,imax,musical,mystery,romance,sci-fi,thriller,war,western
0,1,toy story,Toy Story,1995,0,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2,jumanji,Jumanji,1995,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,grumpier old men,Grumpier Old Men,1995,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
3,4,waiting to exhale,Waiting to Exhale,1995,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
4,5,father of the bride part ii,Father of the Bride Part II,1995,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df_meta = df_movies[['movieId', 'search_title', 'title', 'year']]
df_meta.head()

,movieId,search_title,title,year
0,1,toy story,Toy Story,1995
1,2,jumanji,Jumanji,1995
2,3,grumpier old men,Grumpier Old Men,1995
3,4,waiting to exhale,Waiting to Exhale,1995
4,5,father of the bride part ii,Father of the Bride Part II,1995


In [4]:
X_genres = df_movies.drop(columns=['movieId', 'search_title', 'title', 'year'])
X_genres.head()

,action,adventure,animation,children,comedy,crime,documentary,drama,fantasy,film-noir,horror,imax,musical,mystery,romance,sci-fi,thriller,war,western
0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### 1.2 Similarity Matrix Computation

In [5]:
similarity_matrix = cosine_similarity(X_genres)
similarity_matrix

array([[1.        , 0.77459667, 0.31622777, ..., 0.51639778, 0.        ,
        0.        ],
       [0.77459667, 1.        , 0.        , ..., 0.33333333, 0.        ,
        0.        ],
       [0.31622777, 0.        , 1.        , ..., 0.40824829, 0.        ,
        0.        ],
       ...,
       [0.51639778, 0.33333333, 0.40824829, ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.57735027],
       [0.        , 0.        , 0.        , ..., 0.        , 0.57735027,
        1.        ]], shape=(5230, 5230))

### 1.3 Recommendation Engine

In [6]:
def get_content_recommendations(query_title, similarity_matrix, top_n=5):
    # Find movies matching the search title
    matched_indices = df_meta[
        df_meta['search_title'].str.contains(query_title.lower(), na=False)
    ].index

    if matched_indices.empty:
        print(f'Movie "{query_title}" not found in the catalog.')
        return None
    
    movie_idx = matched_indices[0]

    print(f'Recommendations based on: "{df_meta["title"][movie_idx]}"')

    # Retrieve similarity scores for the selected movie
    similarity_scores = similarity_matrix[movie_idx]

    # Sort movies by similarity in descending order
    sorted_movie_indices = similarity_scores.argsort()[::-1]

    # Remove the queried movie from the recommendations
    recommended_indices = [
        idx for idx in sorted_movie_indices if idx != movie_idx
    ]
    
    top_indices = recommended_indices[:top_n]

    # Build the output DataFrame
    recommendations_df = df_meta.iloc[top_indices].copy()

    recommendations_df['similarity_score'] = similarity_scores[top_indices]

    return recommendations_df

### 1.4 Testing the Recommendations

In [7]:
query_title = 'Toy Story'
recommended_movies = get_content_recommendations(query_title, similarity_matrix, top_n=5)
recommended_movies

Recommendations based on: "Toy Story"


,movieId,search_title,title,year,similarity_score
2028,3114,toy story 2,Toy Story 2,1999,1.0
4220,53121,shrek the third,Shrek the Third,2007,1.0
2816,4886,"monsters, inc.","Monsters, Inc.",2001,1.0
1502,2294,antz,Antz,1998,1.0
2482,4016,the emperor's new groove,The Emperor's New Groove,2000,1.0


In [8]:
query_title = 'Amityville'
recommended_movies = get_content_recommendations(query_title, similarity_matrix, top_n=5)
recommended_movies

Recommendations based on: "The Amityville Horror"


,movieId,search_title,title,year,similarity_score
4457,62733,quarantine,Quarantine,2008,1.0
3807,27338,the hole,The Hole,2001,1.0
3839,27839,the ring two,The Ring Two,2005,1.0
2771,4754,the wicker man,The Wicker Man,1973,1.0
4663,74228,triangle,Triangle,2009,1.0


In [9]:
query_title = 'Matrix'
recommended_movies = get_content_recommendations(query_title, similarity_matrix, top_n=5)
recommended_movies

Recommendations based on: "The Matrix"


,movieId,search_title,title,year,similarity_score
5211,130490,insurgent,Insurgent,2015,1.0
3117,5903,equilibrium,Equilibrium,2002,1.0
837,1240,the terminator,The Terminator,1984,1.0
65,76,screamers,Screamers,1995,1.0
4614,71530,surrogates,Surrogates,2009,1.0
